# TrDGL-FuzzVn: B3 stop-token repair and Q3/Q4 validation on Snowflake

This notebook is a **diagnostic and continuation artifact**, not a silent replacement for the frozen campaign. It:

1. fetches the staged tuned Q3_K_M and Q4_K_M GGUF files;
2. inspects tokenizer/chat metadata and probes raw generation;
3. compares the original chat path with an explicit string-stop serving wrapper;
4. writes machine-readable Q3/Q4 evidence to a Snowflake stage;
5. optionally executes the remaining B3 tasks for seeds `12011`, `19001`, and `27103` when the frozen task JSONL is supplied.

The original B0-B3 five-seed campaign remains valid only when all baselines use the same frozen manifest, prompt budget, decoding settings, evaluator, and order contract. Q4 is reported as a **quantization sensitivity diagnostic**, not merged into the original Q3 campaign denominator.


In [ ]:
# Snowflake Notebook setup: run in a Container Runtime with sufficient memory.
# No credentials are embedded; the notebook uses its active Snowflake session.
from snowflake.snowpark.context import get_active_session
from pathlib import Path
import os, json, re, hashlib, time, platform, subprocess, sys

session = get_active_session()
WORK = Path('/tmp/trdgl_b3_q3_q4')
WORK.mkdir(parents=True, exist_ok=True)
RESULTS = WORK / 'results'
RESULTS.mkdir(exist_ok=True)

STAGE_ROOT = '@AI_LAB.PHASE_A.TRDGL_GGUF_STAGE'
Q3_STAGE = STAGE_ROOT + '/models/trdgl_gemma4_26b_a4b_qa/rev_7e25063a3755/gemma4-26b-a4b-qa.Q3_K_M.gguf'
Q4_STAGE = STAGE_ROOT + '/models/trdgl_gemma4_26b_a4b_qa/rev_7e25063a3755/q4_k_m/gemma4-26b-a4b-qa.Q4_K_M.gguf'
OUTPUT_STAGE = STAGE_ROOT + '/experiments/b3_stopfix_snowflake/'

EXPECTED = {
    'Q3_K_M': {'bytes': 13286716704, 'sha256': '2f35a80c395f31c1b39003d1e4e14df6b06796f343bb670c37d53828a96e482b'},
    'Q4_K_M': {'bytes': 16795999008, 'sha256': 'ea4a99b6dbdd3fedc9c20838e22dd2fcf272ed0d84360d42deaf0b15e955a068'},
}
print({'python': platform.python_version(), 'platform': platform.platform(), 'work': str(WORK)})


In [ ]:
# Install runtime dependencies. In a restricted Snowflake runtime, add these packages
# through Notebook Packages or a custom runtime image instead of pip.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'llama-cpp-python>=0.3.23,<0.4', 'pandas>=2.0', 'nbformat>=5.9'
])
from llama_cpp import Llama
import pandas as pd
print('llama-cpp-python installed')


In [ ]:
def sha256_file(path: Path, chunk=16*1024*1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def stage_get(remote: str, local_dir: Path) -> Path:
    local_dir.mkdir(parents=True, exist_ok=True)
    rows = session.file.get(remote, f'file://{local_dir}', parallel=16)
    if not rows:
        raise RuntimeError(f'GET returned no rows for {remote}')
    name = remote.rsplit('/', 1)[-1]
    path = local_dir / name
    if not path.exists():
        matches = list(local_dir.rglob(name))
        if len(matches) != 1:
            raise FileNotFoundError((remote, matches))
        path = matches[0]
    return path

models = {}
for label, remote in [('Q3_K_M', Q3_STAGE), ('Q4_K_M', Q4_STAGE)]:
    p = stage_get(remote, WORK / label)
    observed = {'bytes': p.stat().st_size, 'sha256': sha256_file(p)}
    if observed != EXPECTED[label]:
        raise RuntimeError(f'{label} integrity mismatch: {observed} != {EXPECTED[label]}')
    models[label] = p
    print(label, observed, p)


In [ ]:
STOP_STRINGS = ['<end_of_turn>', '<start_of_turn>', '<eos>']
MARKER_RE = re.compile(
    r'(?:<)?(?:start_of_turn|end_of_turn|eos)(?:>)?|'
    r'<\|turn>|<turn\|>|'
    r'^(?:model|assistant)\s*[:\n]+',
    flags=re.IGNORECASE | re.MULTILINE,
)

def clean_b3_text(text: str) -> str:
    # Cut at the first full or orphaned turn marker, then remove residues.
    cut = re.search(r'(?:<)?(?:start_of_turn|end_of_turn|eos)(?:>)?', text, re.I)
    if cut:
        text = text[:cut.start()]
    text = MARKER_RE.sub('', text)
    return text.strip()

class B3Chat:
    def __init__(self, llm: Llama):
        self.llm = llm

    @staticmethod
    def render(messages):
        # Match the literal training format discovered in the tuned artifact.
        parts = []
        for m in messages:
            role = 'model' if m['role'] == 'assistant' else m['role']
            parts.append(f'<start_of_turn>{role}\n{m["content"]}<end_of_turn>')
        parts.append('<start_of_turn>model\n')
        return ''.join(parts)

    def complete(self, messages, *, seed=3407, max_tokens=600, temperature=0.2,
                 top_p=0.95, top_k=40, repeat_penalty=1.1):
        prompt = self.render(messages)
        out = self.llm.create_completion(
            prompt=prompt,
            seed=seed,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repeat_penalty=repeat_penalty,
            stop=STOP_STRINGS,
            echo=False,
        )
        raw = out['choices'][0]['text']
        return {
            'text': clean_b3_text(raw),
            'raw_text': raw,
            'finish_reason': out['choices'][0].get('finish_reason'),
            'usage': out.get('usage', {}),
        }


In [ ]:
# Focused before/after matrix. Keep prompts deterministic and small enough to run twice per quantization.
PROBES = [
    'Reply with only the number: 17 * 19.',
    'Write one valid Python assertion that checks torch.add(torch.tensor(2), torch.tensor(3)) equals 5.',
    'Give one concise sentence explaining what torch.reshape does.',
]

def load_model(path: Path):
    return Llama(
        model_path=str(path),
        n_ctx=4096,
        n_threads=max(1, min(16, os.cpu_count() or 4)),
        n_batch=256,
        verbose=False,
    )

def has_marker(text):
    return bool(re.search(r'(?:start_of_turn|end_of_turn|<\|turn>|<turn\|>)', text, re.I))

def run_probe(label, path):
    llm = load_model(path)
    fixed = B3Chat(llm)
    rows = []
    try:
        for i, prompt in enumerate(PROBES):
            seed = 3407 + i
            before = llm.create_chat_completion(
                messages=[{'role':'user','content':prompt}],
                seed=seed, max_tokens=160, temperature=0.2, top_p=0.95, top_k=40,
            )
            bchoice = before['choices'][0]
            btext = bchoice['message']['content'] or ''
            after = fixed.complete([{'role':'user','content':prompt}], seed=seed, max_tokens=160)
            rows.extend([
                {'quant':label,'probe':i,'mode':'before','finish_reason':bchoice.get('finish_reason'),
                 'marker_present':has_marker(btext),'clean':not has_marker(btext),'text':btext},
                {'quant':label,'probe':i,'mode':'after','finish_reason':after['finish_reason'],
                 'marker_present':has_marker(after['text']),'clean':not has_marker(after['text']),'text':after['text']},
            ])
    finally:
        del fixed, llm
        import gc; gc.collect()
    return rows

probe_rows=[]
for label, path in models.items():
    probe_rows.extend(run_probe(label, path))
probe_df=pd.DataFrame(probe_rows)
display(probe_df[['quant','probe','mode','finish_reason','marker_present','clean','text']])


In [ ]:
# Fail closed: the repaired path must be clean for every probe and both quantizations.
after = probe_df[probe_df['mode'] == 'after']
assert len(after) == len(PROBES) * 2
assert after['clean'].all(), after[~after['clean']]
assert set(after['finish_reason']) <= {'stop', 'eos'}, set(after['finish_reason'])

summary = {
    'schema_version': 'trdgl_b3_stopfix_snowflake_v1',
    'created_utc': pd.Timestamp.utcnow().isoformat(),
    'stage_models': {'Q3_K_M': Q3_STAGE, 'Q4_K_M': Q4_STAGE},
    'model_integrity': EXPECTED,
    'stop_strings': STOP_STRINGS,
    'probe_count_per_quant': len(PROBES),
    'results': probe_df.groupby(['quant','mode']).agg(
        rows=('clean','size'), clean=('clean','sum'), marker_rows=('marker_present','sum')
    ).reset_index().to_dict('records'),
    'interpretation': (
        'Q3/Q4 agreement supports a serving stop/template mismatch rather than a quantization-specific collapse. '
        'This is diagnostic evidence and does not replace the frozen campaign.'
    ),
}
(RESULTS/'probe_results.jsonl').write_text(
    ''.join(json.dumps(r, ensure_ascii=False)+'\n' for r in probe_rows), encoding='utf-8')
(RESULTS/'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
summary


## Optional: remaining B3 seed continuation

The following cell is intentionally gated. It runs only when a frozen task file is staged at
`@AI_LAB.PHASE_A.TRDGL_GGUF_STAGE/experiments/frozen_inputs/remaining_b3_tasks.jsonl`.
Each line must contain at least `task_id`, `generation_seed`, `prompt`, and `prompt_hash`.

This produces **separate Q3 and Q4 diagnostic streams**. Do not combine Q4 rows with the original Q3 B0-B3 campaign. A full paper result still requires compatible B0, B1, B2, and B3 shards under the frozen harness.


In [ ]:
RUN_REMAINING_B3 = False
TASK_STAGE = STAGE_ROOT + '/experiments/frozen_inputs/remaining_b3_tasks.jsonl'
REMAINING_SEEDS = {12011, 19001, 27103}

if RUN_REMAINING_B3:
    task_path = stage_get(TASK_STAGE, WORK/'tasks')
    tasks = [json.loads(x) for x in task_path.read_text(encoding='utf-8').splitlines() if x.strip()]
    tasks = [t for t in tasks if t['generation_seed'] in REMAINING_SEEDS]
    if not tasks:
        raise RuntimeError('No remaining frozen B3 tasks found')
    required = {'task_id','generation_seed','prompt','prompt_hash'}
    for t in tasks:
        if not required <= set(t):
            raise ValueError(f'Missing task fields: {required-set(t)}')
        observed = hashlib.sha256(t['prompt'].encode('utf-8')).hexdigest()
        if observed != t['prompt_hash']:
            raise ValueError(f'Prompt hash mismatch: {t["task_id"]}')

    for label, path in models.items():
        out_path = RESULTS/f'remaining_b3_{label}.jsonl'
        completed=set()
        if out_path.exists():
            completed={json.loads(x)['task_id'] for x in out_path.read_text().splitlines() if x.strip()}
        llm=load_model(path); chat=B3Chat(llm)
        try:
            with out_path.open('a', encoding='utf-8') as out:
                for n,t in enumerate(tasks,1):
                    if t['task_id'] in completed: continue
                    tic=time.time()
                    result=chat.complete([{'role':'user','content':t['prompt']}], seed=t['generation_seed'])
                    row={
                        'schema_version':'trdgl_b3_stopfix_event_v1',
                        'quant':label,
                        'task_id':t['task_id'],
                        'generation_seed':t['generation_seed'],
                        'prompt_hash':t['prompt_hash'],
                        'raw_output_hash':hashlib.sha256(result['raw_text'].encode()).hexdigest(),
                        'text':result['text'],
                        'finish_reason':result['finish_reason'],
                        'marker_present':has_marker(result['text']),
                        'wall_seconds':time.time()-tic,
                    }
                    out.write(json.dumps(row,ensure_ascii=False)+'\n'); out.flush()
                    if n % 10 == 0: print(label,n,'/',len(tasks))
        finally:
            del chat,llm
            import gc; gc.collect()


In [ ]:
# Upload all machine-readable outputs to the experiment prefix.
manifest=[]
for p in sorted(RESULTS.glob('*')):
    if not p.is_file(): continue
    manifest.append({'name':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
(RESULTS/'MANIFEST_SHA256.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
put_results=session.file.put(str(RESULTS/'*'), OUTPUT_STAGE, parallel=16, auto_compress=False, overwrite=True)
listed=session.sql(f'LIST {OUTPUT_STAGE}').collect()
print('uploaded',len(put_results),'files; stage entries',len(listed))
for r in listed: print(r)


## Interpretation contract

- A clean Q3 and Q4 repaired path supports the **string-level stopping mismatch** explanation.
- Similar Q3/Q4 behavior argues against quantization level being the primary cause of collapse.
- This notebook does not establish that the tokenizer control IDs are exactly 105/106 unless the runtime metadata/probe output records that fact.
- Q4 results are a sensitivity analysis and must not be pooled with the frozen Q3 campaign.
- Final paper completion still requires the remaining compatible B0-B3 shards, external comparator, full Atlas intervention, and Linux compiled numerical matrix listed in `SUBMISSION_GAPS.md`.
